# IDAP 2026 camera-ready — Colab #2: multi-seed retraining

Closes reviewer comment **R1** (*"only a single training run was conducted"*) and the LoRA half of **R2**.

Retrains every **trained** system in Table I at seeds 1337 and 2024 (seed 42 already exists), plus LoRA rank 16 and 32 so the *"rank 8 is Pareto-optimal"* claim can be tested instead of asserted.

| group | runs | approx |
|---|---|---|
| `core` | B2-human, S-gpt, S-claude x 2 seeds | 6 x ~30 min = **3.0 h** |
| `lora` | S-gpt r16, r32 x 2 seeds | 4 x ~30 min = **2.0 h** |
| `infer` | 10 checkpoints on MLSUM + 6 on TR-News | **~1.7 h** |

**Run the groups in separate sessions.** Every step is idempotent — finished runs are skipped and each checkpoint is copied to Drive as soon as it lands, so a disconnect costs at most one run.

> Runtime → Change runtime type → **T4 GPU**.


In [ ]:
import os, subprocess, time
print(subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'],
                     capture_output=True, text=True).stdout.strip() or 'NO GPU — switch runtime to T4')
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/ceng467_termproject/CENG467-term-project'
assert os.path.isdir(DRIVE), DRIVE

REPO, BRANCH = '/content/repo', 'revision-v2'
URL = 'https://github.com/cagancaliskan/CENG467-term-project.git'
if os.path.isdir(REPO):
    !cd {REPO} && git fetch origin {BRANCH} -q && git checkout {BRANCH} -q && git pull -q
else:
    !git clone -q --branch {BRANCH} {URL} {REPO}
os.chdir(REPO)
!git log --oneline -1
!pip install -q 'transformers>=4.50.0,<5.0.0' 'peft>=0.13.0' 'datasets>=2.20.0,<3.0' \
               'accelerate>=1.1.0' 'sentencepiece>=0.2.0' 2>&1 | tail -3

# Colab ships torchao 0.10; peft's LoRA dispatcher calls is_torchao_available()
# and RAISES when the version is below its minimum instead of skipping the
# backend. This project uses no quantized adapters, so removing torchao is the
# clean fix -- upgrading it risks dragging a different torch build along.
!pip uninstall -y -q torchao 2>&1 | tail -1
import subprocess as _sp
_chk = _sp.run(['python','-c',
    'from peft.import_utils import is_torchao_available as f; print("torchao gate:", f())'],
    capture_output=True, text=True)
print(_chk.stdout.strip() or _chk.stderr.strip()[-300:])
assert 'torchao gate: False' in _chk.stdout, 'torchao still blocking peft — tell me before continuing'


## Step 0 — consolidate the teacher cache (run once, ~5 min)

The paper's Sec. VII-C says the synthetic caches are publicly available on GitHub. They are not: `git ls-files data/synthetic` returns nothing, and the cache exists only in Drive as ~22k individual `.json` files.

Consolidating each (teacher, prompt) pair into one JSONL fixes both problems at once — the files become small enough to commit, so the claim becomes true, and every later Colab session copies four files instead of twenty-two thousand.

Skip this cell if `data/synthetic/*.jsonl` already exists on Drive.


In [ ]:
import glob, shutil, os
existing = glob.glob(f'{DRIVE}/data/synthetic/*.jsonl')
if existing:
    print('already consolidated:'); [print('  ', os.path.basename(p)) for p in existing]
else:
    os.makedirs('data/synthetic', exist_ok=True)
    print('copying per-article cache from Drive (slow, one time only)...')
    shutil.copytree(f'{DRIVE}/data/synthetic', 'data/synthetic', dirs_exist_ok=True)
    !python scripts/consolidate_synthetic.py --root data/synthetic
    for j in glob.glob('data/synthetic/*.jsonl'):
        shutil.copy2(j, f'{DRIVE}/data/synthetic/' + os.path.basename(j))
    print('\nconsolidated JSONLs copied back to Drive.')

print()
print('Download these four files and commit them to the repo so the paper\'s')
print('reproducibility claim is actually true:')
for p in sorted(glob.glob(f'{DRIVE}/data/synthetic/*.jsonl')):
    print(f'   {os.path.basename(p):<32} {os.path.getsize(p)/1e6:6.1f} MB')


## What is about to run

`--dry-run` prints the exact commands without executing them. Check the run names and seeds look right before spending GPU hours.

Note: `prepare_synthetic` is always called with **seed 42**, never the training seed. The training subset must stay fixed across seeds, otherwise the measured spread mixes optimisation noise with data resampling and the whole point of the exercise is lost (`PREREGISTRATION.md`, section 5).


In [ ]:
!python scripts/05_multiseed.py --group all --seeds 1337,2024 --drive {DRIVE} --dry-run


## Group 1 — core systems (~3.0 h)

B2-human, S-gpt, S-claude at seeds 1337 and 2024. These three are the trained rows of Table I; without them R1 stays open.


In [ ]:
!python scripts/05_multiseed.py --group core --seeds 1337,2024 --drive {DRIVE}


## Group 2 — LoRA rank (~2.0 h)

S-gpt at rank 16 and 32, seeds 1337 and 2024. Rank 8 at all three seeds comes from group 1. This is what lets the paper say whether r8 and r16 are actually distinguishable.


In [ ]:
!python scripts/05_multiseed.py --group lora --seeds 1337,2024 --drive {DRIVE}


## Group 3 — inference (~1.7 h)

All new checkpoints on MLSUM-TR test; the three core systems also on TR-News. Uses the fixed decoder, so these predictions are sentinel-free from the start.


In [ ]:
!python scripts/05_multiseed.py --group infer --seeds 1337,2024 --drive {DRIVE}


## Status


In [ ]:
import glob, json
print('checkpoints on Drive:')
for p in sorted(glob.glob(f'{DRIVE}/outputs/checkpoints/*_s*/final/adapter_config.json')):
    print('  ', p.split('/checkpoints/')[1].split('/final')[0])
print()
print('multiseed predictions on Drive:')
for p in sorted(glob.glob(f'{DRIVE}/outputs/predictions/multiseed/*.jsonl')):
    n = sum(1 for _ in open(p, encoding='utf-8'))
    print(f'   {os.path.basename(p):<40} {n:>5} rows')
print()
print('When all three groups are done, tell me and I will send the scoring + statistics notebook.')
